<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/COMPLETE_CBP_TOPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.init as init

# ==============================================================================
# Dynamic Derivation of Primes and Lambda
# ==============================================================================
def derive_primes_and_lambda(max_prime=13):
    is_prime = [True] * (max_prime + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(math.isqrt(max_prime)) + 1):
        if is_prime[p]:
            for i in range(p * p, max_prime + 1, p):
                is_prime[i] = False
    primes = [p for p, valid in enumerate(is_prime) if valid]

    product = 1.0
    for p in primes:
        product *= (1.0 - (1.0 / math.sqrt(p)))

    return primes, 1.0 - product

PRIME_ANCHORS, LAMBDA_DERIVED = derive_primes_and_lambda(max_prime=13)
print(f"Dynamically Derived Primes : {PRIME_ANCHORS}")
print(f"Dynamically Derived Lambda : {LAMBDA_DERIVED:.10f}")


# ==============================================================================
# Topological Governor with Weight-Decay Invariance Lock
# ==============================================================================
class TopologicalGovernor:
    def __init__(self, prime_indices, lam):
        self.prime_indices = prime_indices
        self.lam = lam

    def pre_step(self, model: nn.Module):
        """Attenuates plastic gradient subspace and clamps prime anchor gradients."""
        with torch.no_grad():
            for param in model.parameters():
                if param.grad is None:
                    continue

                param.grad.mul_(self.lam)

                dim = param.dim()
                if dim >= 2:
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid, :] = 0.0
                elif dim == 1:
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid] = 0.0

    def post_step(self, model: nn.Module, initial_anchors: dict):
        """Locks prime coordinate values against optimizer weight decay drift."""
        with torch.no_grad():
            model.layer1.linear.weight[self.prime_indices, :] = initial_anchors['l1']
            model.layer2.linear.weight[self.prime_indices, :] = initial_anchors['l2']


# ==============================================================================
# Continual Linear Layer (CBP with Prime Shielding)
# ==============================================================================
class ContinualLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, replacement_rate: float = 1e-4,
                 maturity: int = 100, eta: float = 0.99):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.replacement_rate = replacement_rate
        self.maturity = maturity
        self.eta = eta

        self.linear = nn.Linear(in_features, out_features)
        init.kaiming_uniform_(self.linear.weight, nonlinearity='relu')
        init.zeros_(self.linear.bias)

        self.register_buffer('utility', torch.zeros(out_features))
        self.register_buffer('age', torch.zeros(out_features, dtype=torch.long))
        self.register_buffer('accumulated_replacements', torch.tensor(0.0))
        self.last_activation = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.linear(x)
        self.last_activation = torch.relu(h)
        return self.last_activation

    def update_utility(self, next_layer_weight: torch.Tensor):
        with torch.no_grad():
            if self.last_activation is None:
                return
            mean_act = torch.mean(torch.abs(self.last_activation), dim=0)
            outgoing_weight_sum = torch.sum(torch.abs(next_layer_weight), dim=0)
            instantaneous_utility = mean_act * outgoing_weight_sum
            self.utility.mul_(self.eta).add_(instantaneous_utility, alpha=(1.0 - self.eta))
            self.age.add_(1)

    def reinitialize_units(self, next_layer_linear: nn.Linear, protected_indices=None):
        with torch.no_grad():
            eligible_mask = self.age >= self.maturity
            if protected_indices:
                for idx in protected_indices:
                    if idx < self.out_features:
                        eligible_mask[idx] = False

            n_eligible = eligible_mask.sum().item()
            if n_eligible == 0:
                return

            self.accumulated_replacements.add_(n_eligible * self.replacement_rate)

            while self.accumulated_replacements >= 1.0:
                masked_utility = torch.where(
                    eligible_mask,
                    self.utility,
                    torch.tensor(float('inf'), device=self.utility.device)
                )
                r = torch.argmin(masked_utility).item()

                bound = (1.0 / self.in_features) ** 0.5
                init.uniform_(self.linear.weight[r, :], -bound, bound)
                self.linear.bias[r].zero_()

                next_layer_linear.weight[:, r].zero_()

                self.utility[r] = 0.0
                self.age[r] = 0
                eligible_mask[r] = False
                self.accumulated_replacements.sub_(1.0)


# ==============================================================================
# Governed Network
# ==============================================================================
class GovernedContinualMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int,
                 replacement_rate: float = 1e-3, maturity: int = 50):
        super().__init__()
        self.layer1 = ContinualLinear(in_dim, hidden_dim, replacement_rate, maturity)
        self.layer2 = ContinualLinear(hidden_dim, hidden_dim, replacement_rate, maturity)
        self.head = nn.Linear(hidden_dim, out_dim)
        init.kaiming_uniform_(self.head.weight, nonlinearity='linear')
        init.zeros_(self.head.bias)

        self.governor = TopologicalGovernor(PRIME_ANCHORS, LAMBDA_DERIVED)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h1 = self.layer1(x)
        h2 = self.layer2(h1)
        return self.head(h2)

    def govern_pre_step(self):
        self.governor.pre_step(self)

    def govern_post_step(self, initial_anchors: dict):
        self.governor.post_step(self, initial_anchors)

    def cbp_step(self):
        self.layer1.update_utility(self.layer2.linear.weight)
        self.layer2.update_utility(self.head.weight)
        self.layer1.reinitialize_units(self.layer2.linear, protected_indices=PRIME_ANCHORS)
        self.layer2.reinitialize_units(self.head, protected_indices=PRIME_ANCHORS)

    def verify_prime_invariance(self, initial_anchors: dict) -> float:
        with torch.no_grad():
            w1_drift = torch.max(torch.abs(self.layer1.linear.weight[PRIME_ANCHORS, :] - initial_anchors['l1'])).item()
            w2_drift = torch.max(torch.abs(self.layer2.linear.weight[PRIME_ANCHORS, :] - initial_anchors['l2'])).item()
            return max(w1_drift, w2_drift)


# ==============================================================================
# Online Verification Run
# ==============================================================================
if __name__ == "__main__":
    torch.manual_seed(42)

    in_dim, hidden_dim, out_dim = 64, 128, 10
    model = GovernedContinualMLP(in_dim, hidden_dim, out_dim, replacement_rate=1e-3, maturity=30)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    initial_anchors = {
        'l1': model.layer1.linear.weight[PRIME_ANCHORS, :].clone(),
        'l2': model.layer2.linear.weight[PRIME_ANCHORS, :].clone()
    }

    print("\n--- Executing Invariance Verification ---")
    current_center = torch.zeros(in_dim)

    for step in range(1, 1001):
        if step % 250 == 1:
            current_center = torch.randn(in_dim)
            task_id = (step // 250) + 1
            print(f"\n[Environment Shift] Task {task_id} Active at Step {step}")

        x = torch.randn(32, in_dim) + current_center
        y = torch.randint(0, out_dim, (32,))

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()

        # Enforce gradient attenuation & clamp prime gradients
        model.govern_pre_step()
        optimizer.step()
        # Enforce exact coordinate retention against weight decay
        model.govern_post_step(initial_anchors)
        # Continual Backprop step with shielded primes
        model.cbp_step()

        if step % 250 == 0:
            drift = model.verify_prime_invariance(initial_anchors)
            l1_u = model.layer1.utility.mean().item()
            l2_u = model.layer2.utility.mean().item()
            print(f"Step {step:4d} | Loss: {loss.item():.4f} | Prime Anchor Drift: {drift:.10f} | Utility (L1/L2): {l1_u:.4f} / {l2_u:.4f}")

Dynamically Derived Primes : [2, 3, 5, 7, 11, 13]
Dynamically Derived Lambda : 0.9785142874

--- Executing Invariance Verification ---

[Environment Shift] Task 1 Active at Step 1
Step  250 | Loss: 2.3079 | Prime Anchor Drift: 0.0000000000 | Utility (L1/L2): 9.0760 / 0.4613

[Environment Shift] Task 2 Active at Step 251
Step  500 | Loss: 2.3874 | Prime Anchor Drift: 0.0000000000 | Utility (L1/L2): 10.2141 / 0.4392

[Environment Shift] Task 3 Active at Step 501
Step  750 | Loss: 2.4041 | Prime Anchor Drift: 0.0000000000 | Utility (L1/L2): 10.2805 / 0.3516

[Environment Shift] Task 4 Active at Step 751
Step 1000 | Loss: 2.4367 | Prime Anchor Drift: 0.0000000000 | Utility (L1/L2): 10.4019 / 0.3778


In [ ]:
!nvidia-smi

Fri Sep 25 23:23:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   62C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## no-topo

In [ ]:
import math
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import load_dataset

# ==============================================================================
# 0. Deterministic Setup (Identical Seed 123)
# ==============================================================================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Same prime coordinate probe set to monitor unconstrained erosion
PRIME_PROBES = [2, 3, 5, 7, 11, 13]

# ==============================================================================
# 1. Unconstrained Baseline Streaming (Without Topological Governor)
# ==============================================================================
def run_baseline_llm_stream(total_steps=5000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = "Qwen/Qwen2.5-0.5B"

    print(f"\nInitializing Baseline (No TOPO) Tokenizer & Model: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    ).to(device)
    model.train()

    target_embed_name = None
    for name, param in model.named_parameters():
        if any(k in name.lower() for k in ["embed_tokens.weight", "wte.weight"]):
            target_embed_name = name
            break

    # Snapshot initial coordinates for drift tracking
    initial_anchors = {}
    for name, param in model.named_parameters():
        if name == target_embed_name:
            initial_anchors[name] = param.data[PRIME_PROBES, :].clone()

    # Standard unconstrained AdamW with identical hyperparameters
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)

    # Identical live stream
    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True
    )

    def text_stream():
        for sample in dataset:
            txt = sample.get("text", "").strip()
            if len(txt) > 80:
                yield txt

    stream_iter = iter(text_stream())

    print(f"\n--- Commencing Baseline Online Streaming (NO GOVERNOR) ({total_steps:,} Steps | Seed {SEED}) ---")

    for step in range(1, total_steps + 1):
        batch = [next(stream_iter) for _ in range(4)]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)
        inputs["labels"] = inputs["input_ids"].clone()

        optimizer.zero_grad()
        loss = model(**inputs).loss
        loss.backward()

        # Standard unconstrained backpropagation: no pre-step, no post-step governor
        optimizer.step()

        # Audit drift on the identical prime probe coordinates
        if step == 1 or step % 250 == 0 or step == total_steps:
            with torch.no_grad():
                current_param = dict(model.named_parameters())[target_embed_name]
                drift = torch.max(
                    torch.abs(current_param[PRIME_PROBES, :] - initial_anchors[target_embed_name])
                ).item()
            print(f"Step {step:5d} / {total_steps} | LM Loss: {loss.item():.4f} | Unconstrained Drift: {drift:.10f}")

    return model, tokenizer

if __name__ == "__main__":
    baseline_model, baseline_tokenizer = run_baseline_llm_stream(total_steps=5000)


Initializing Baseline (No TOPO) Tokenizer & Model: Qwen/Qwen2.5-0.5B


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


--- Commencing Baseline Online Streaming (NO GOVERNOR) (5,000 Steps | Seed 123) ---
Step     1 / 5000 | LM Loss: 3.0187 | Unconstrained Drift: 0.0000305176
Step   250 / 5000 | LM Loss: 3.1067 | Unconstrained Drift: 0.0023651123
Step   500 / 5000 | LM Loss: 2.6103 | Unconstrained Drift: 0.0021667480
Step   750 / 5000 | LM Loss: 2.7175 | Unconstrained Drift: 0.0023345947
Step  1000 / 5000 | LM Loss: 2.6769 | Unconstrained Drift: 0.0023345947
Step  1250 / 5000 | LM Loss: 2.4077 | Unconstrained Drift: 0.0022583008
Step  1500 / 5000 | LM Loss: 2.7927 | Unconstrained Drift: 0.0021972656
Step  1750 / 5000 | LM Loss: 2.9137 | Unconstrained Drift: 0.0021057129
Step  2000 / 5000 | LM Loss: 2.7858 | Unconstrained Drift: 0.0021972656
Step  2250 / 5000 | LM Loss: 3.0223 | Unconstrained Drift: 0.0023803711
Step  2500 / 5000 | LM Loss: 2.8417 | Unconstrained Drift: 0.0022888184
Step  2750 / 5000 | LM Loss: 3.0657 | Unconstrained Drift: 0.0023193359
Step  3000 / 5000 | LM Loss: 2.9372 | Unconstrained

## topo

In [ ]:
import math
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import load_dataset

# ==============================================================================
# 0. Deterministic Setup (Seed 123)
# ==============================================================================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==============================================================================
# 1. Invariant Derivation
# ==============================================================================
def derive_primes_and_lambda(max_prime=13):
    is_prime = [True] * (max_prime + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(math.isqrt(max_prime)) + 1):
        if is_prime[p]:
            for i in range(p * p, max_prime + 1, p):
                is_prime[i] = False
    primes = [p for p, valid in enumerate(is_prime) if valid]

    product = 1.0
    for p in primes:
        product *= (1.0 - (1.0 / math.sqrt(p)))

    return primes, 1.0 - product

PRIME_ANCHORS, LAMBDA_DERIVED = derive_primes_and_lambda(max_prime=13)
print(f"Dynamically Derived Primes : {PRIME_ANCHORS}")
print(f"Dynamically Derived Lambda : {LAMBDA_DERIVED:.10f}")

# ==============================================================================
# 2. Topological Governor
# ==============================================================================
class LLMTopologicalGovernor:
    def __init__(self, prime_indices, lam):
        self.prime_indices = prime_indices
        self.lam = lam

    def pre_step(self, model: torch.nn.Module):
        """Attenuate plastic gradients and clamp prime coordinate gradients."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.grad is None:
                    continue
                param.grad.mul_(self.lam)
                if any(k in name.lower() for k in ["embed", "wte", "lm_head"]):
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid, :] = 0.0

    def post_step(self, model: torch.nn.Module, initial_anchors: dict):
        """Lock prime coordinates against optimizer weight decay drift."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in initial_anchors:
                    param.data[self.prime_indices, :] = initial_anchors[name]

# ==============================================================================
# 3. Train & Return Model Instances (5,000 Steps)
# ==============================================================================
def run_governed_llm_stream(total_steps=5000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = "Qwen/Qwen2.5-0.5B"

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    ).to(device)
    model.train()

    target_embed_name = None
    for name, param in model.named_parameters():
        if any(k in name.lower() for k in ["embed_tokens.weight", "wte.weight"]):
            target_embed_name = name
            break

    if target_embed_name is None:
        raise ValueError("Target embedding tensor not found in model hierarchy.")

    # Snapshot baseline coordinates on device
    initial_anchors = {}
    for name, param in model.named_parameters():
        if name == target_embed_name:
            initial_anchors[name] = param.data[PRIME_ANCHORS, :].clone()

    governor = LLMTopologicalGovernor(PRIME_ANCHORS, LAMBDA_DERIVED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)

    dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

    def text_stream():
        for sample in dataset:
            txt = sample.get("text", "").strip()
            if len(txt) > 80:
                yield txt

    stream_iter = iter(text_stream())

    print(f"\nExecuting {total_steps:,} governed continual streaming steps on {model_id} (Seed {SEED})...\n")
    for step in range(1, total_steps + 1):
        batch = [next(stream_iter) for _ in range(4)]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        inputs["labels"] = inputs["input_ids"].clone()

        optimizer.zero_grad()
        loss = model(**inputs).loss
        loss.backward()

        governor.pre_step(model)
        optimizer.step()
        governor.post_step(model, initial_anchors)

        # Audit Step 1, every 250 steps, and the final step
        if step == 1 or step % 250 == 0 or step == total_steps:
            with torch.no_grad():
                current_param = dict(model.named_parameters())[target_embed_name]
                drift = torch.max(torch.abs(current_param[PRIME_ANCHORS, :] - initial_anchors[target_embed_name])).item()
            print(f"Step {step:5d} / {total_steps} | LM Loss: {loss.item():.4f} | Prime Drift: {drift:.10f}")

    return model, tokenizer

# ==============================================================================
# 4. Local Execution
# ==============================================================================
if __name__ == "__main__":
    model, tokenizer = run_governed_llm_stream(total_steps=5000)
    print("\nTraining session completed. Model and tokenizer are preserved in local memory.")

Dynamically Derived Primes : [2, 3, 5, 7, 11, 13]
Dynamically Derived Lambda : 0.9785142874


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


Executing 5,000 governed continual streaming steps on Qwen/Qwen2.5-0.5B (Seed 123)...

Step     1 / 5000 | LM Loss: 3.0187 | Prime Drift: 0.0000000000
Step   250 / 5000 | LM Loss: 3.1059 | Prime Drift: 0.0000000000
Step   500 / 5000 | LM Loss: 2.6100 | Prime Drift: 0.0000000000
Step   750 / 5000 | LM Loss: 2.7106 | Prime Drift: 0.0000000000
Step  1000 / 5000 | LM Loss: 2.6727 | Prime Drift: 0.0000000000
Step  1250 / 5000 | LM Loss: 2.4076 | Prime Drift: 0.0000000000
Step  1500 / 5000 | LM Loss: 2.7919 | Prime Drift: 0.0000000000
Step  1750 / 5000 | LM Loss: 2.9117 | Prime Drift: 0.0000000000
Step  2000 / 5000 | LM Loss: 2.7832 | Prime Drift: 0.0000000000
Step  2250 / 5000 | LM Loss: 3.0236 | Prime Drift: 0.0000000000
Step  2500 / 5000 | LM Loss: 2.8444 | Prime Drift: 0.0000000000
Step  2750 / 5000 | LM Loss: 3.0683 | Prime Drift: 0.0000000000
Step  3000 / 5000 | LM Loss: 2.9349 | Prime Drift: 0.0000000000
Step  3250 / 5000 | LM Loss: 2.1956 | Prime Drift: 0.0000000000
Step  3500 / 500

In [ ]:
from google.colab import userdata
from huggingface_hub import create_repo

# Retrieve token and set repository target
HF_TOKEN = userdata.get("HF_TOKEN")
user_or_name = "frankmorales2020"
REPO_NAME = "qwen2.5-0.5b-topo-governed-cbp-fineweb"
TARGET_REPO_ID = f"{user_or_name}/{REPO_NAME}"

# Ensure repository exists prior to uploading
create_repo(repo_id=TARGET_REPO_ID, token=HF_TOKEN, repo_type="model", exist_ok=True)

print(f"Pushing weights and configs to https://huggingface.co/{TARGET_REPO_ID}...")

# Push in-memory model weights and configuration
model.push_to_hub(
    repo_id=TARGET_REPO_ID,
    token=HF_TOKEN,
    commit_message="Upload governed model weights (5000 steps, seed 123)"
)

# Push in-memory tokenizer artifacts
tokenizer.push_to_hub(
    repo_id=TARGET_REPO_ID,
    token=HF_TOKEN,
    commit_message="Upload tokenizer configuration"
)

print(f"\nUpload complete: https://huggingface.co/{TARGET_REPO_ID}")

Pushing weights and configs to https://huggingface.co/frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-fineweb...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...hcchgv1/model.safetensors:   2%|1         | 15.9MB /  988MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpiqf9kk2q/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            


Upload complete: https://huggingface.co/frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-fineweb


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-fineweb"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
!nvidia-smi

Fri Sep 25 23:22:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   66C    P0             30W /   72W |    6340MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## mistralai/Mistral-7B-v0.3

In [1]:
import math
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import load_dataset

# ==============================================================================
# 0. Deterministic Setup (Seed 123)
# ==============================================================================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ==============================================================================
# 1. Invariant Derivation
# ==============================================================================
def derive_primes_and_lambda(max_prime=13):
    is_prime = [True] * (max_prime + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(math.isqrt(max_prime)) + 1):
        if is_prime[p]:
            for i in range(p * p, max_prime + 1, p):
                is_prime[i] = False
    primes = [p for p, valid in enumerate(is_prime) if valid]

    product = 1.0
    for p in primes:
        product *= (1.0 - (1.0 / math.sqrt(p)))

    return primes, 1.0 - product

PRIME_ANCHORS, LAMBDA_DERIVED = derive_primes_and_lambda(max_prime=13)
print(f"Dynamically Derived Primes : {PRIME_ANCHORS}")
print(f"Dynamically Derived Lambda : {LAMBDA_DERIVED:.10f}")

# ==============================================================================
# 2. Mistral-7B Topological Governor
# ==============================================================================
class MistralTopologicalGovernor:
    def __init__(self, prime_indices, lam):
        self.prime_indices = prime_indices
        self.lam = lam

    def pre_step(self, model: torch.nn.Module):
        """Attenuate plastic gradients and clamp prime coordinate gradients."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.grad is None:
                    continue
                param.grad.mul_(self.lam)
                # Mistral targets: embed_tokens or lm_head
                if any(k in name.lower() for k in ["embed_tokens", "lm_head"]):
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid, :] = 0.0

    def post_step(self, model: torch.nn.Module, initial_anchors: dict):
        """Lock prime coordinates against AdamW weight decay drift."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in initial_anchors:
                    param.data[self.prime_indices, :] = initial_anchors[name]

# ==============================================================================
# 3. Execution Loop for mistralai/Mistral-7B-v0.3
# ==============================================================================
def run_mistral_experiment(mode="topo", total_steps=5000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = "mistralai/Mistral-7B-v0.3"

    print(f"\nInitializing Mistral-7B-v0.3 ({mode.upper()}) | Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load in bfloat16 with flash attention / memory optimizations
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model.train()
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    target_embed_name = None
    for name, param in model.named_parameters():
        if "embed_tokens.weight" in name.lower():
            target_embed_name = name
            break

    if target_embed_name is None:
        raise ValueError("Mistral embedding matrix 'embed_tokens.weight' not located.")

    # Snapshot anchor coordinates
    initial_anchors = {}
    for name, param in model.named_parameters():
        if name == target_embed_name:
            initial_anchors[name] = param.data[PRIME_ANCHORS, :].clone()

    governor = MistralTopologicalGovernor(PRIME_ANCHORS, LAMBDA_DERIVED) if mode == "topo" else None
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)

    dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

    def text_stream():
        for sample in dataset:
            txt = sample.get("text", "").strip()
            if len(txt) > 80:
                yield txt

    stream_iter = iter(text_stream())

    print(f"\n--- Commencing Mistral-7B Streaming [{mode.upper()}] ({total_steps:,} Steps | Seed {SEED}) ---")

    for step in range(1, total_steps + 1):
        batch = [next(stream_iter) for _ in range(2)]  # Batch size 2 for 7B scale on single-GPU
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        inputs["labels"] = inputs["input_ids"].clone()

        optimizer.zero_grad()
        loss = model(**inputs).loss
        loss.backward()

        if mode == "topo":
            governor.pre_step(model)
            optimizer.step()
            governor.post_step(model, initial_anchors)
        else:
            optimizer.step()

        # Audit drift on probe indices
        if step == 1 or step % 250 == 0 or step == total_steps:
            with torch.no_grad():
                current_param = dict(model.named_parameters())[target_embed_name]
                drift = torch.max(torch.abs(current_param[PRIME_ANCHORS, :] - initial_anchors[target_embed_name])).item()
            print(f"Step {step:5d} / {total_steps} | LM Loss: {loss.item():.4f} | Coordinate Drift: {drift:.10f}")

    return model, tokenizer

if __name__ == "__main__":
    # Execute Governed Run (Toggle mode='baseline' for unconstrained run)
    model, tokenizer = run_mistral_experiment(mode="topo", total_steps=5000)

Dynamically Derived Primes : [2, 3, 5, 7, 11, 13]
Dynamically Derived Lambda : 0.9785142874

Initializing Mistral-7B-v0.3 (TOPO) | Hardware: NVIDIA A100-SXM4-80GB


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


--- Commencing Mistral-7B Streaming [TOPO] (5,000 Steps | Seed 123) ---


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step     1 / 5000 | LM Loss: 2.3570 | Coordinate Drift: 0.0000000000
Step   250 / 5000 | LM Loss: 1.8102 | Coordinate Drift: 0.0000000000
Step   500 / 5000 | LM Loss: 2.9918 | Coordinate Drift: 0.0000000000
Step   750 / 5000 | LM Loss: 2.3262 | Coordinate Drift: 0.0000000000
Step  1000 / 5000 | LM Loss: 1.8481 | Coordinate Drift: 0.0000000000
Step  1250 / 5000 | LM Loss: 2.2273 | Coordinate Drift: 0.0000000000
Step  1500 / 5000 | LM Loss: 2.2400 | Coordinate Drift: 0.0000000000
Step  1750 / 5000 | LM Loss: 3.1171 | Coordinate Drift: 0.0000000000
Step  2000 / 5000 | LM Loss: 2.0805 | Coordinate Drift: 0.0000000000
Step  2250 / 5000 | LM Loss: 2.6376 | Coordinate Drift: 0.0000000000
Step  2500 / 5000 | LM Loss: 1.9101 | Coordinate Drift: 0.0000000000
Step  2750 / 5000 | LM Loss: 2.3600 | Coordinate Drift: 0.0000000000
Step  3000 / 5000 | LM Loss: 2.3357 | Coordinate Drift: 0.0000000000
Step  3250 / 5000 | LM Loss: 2.5071 | Coordinate Drift: 0.0000000000
Step  3500 / 5000 | LM Loss: 2.309

## moe

In [1]:
!pip install -U bitsandbytes>=0.46.1 -q

In [1]:
!nvidia-smi

Sat Sep 26 07:31:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   38C    P0             50W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import math
import random
import os
import gc
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from datasets import load_dataset

# Eliminate CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ==============================================================================
# 0. Deterministic Setup (Seed 123)
# ==============================================================================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

gc.collect()
torch.cuda.empty_cache()

# ==============================================================================
# 1. Invariant Coordinate Ring Derivation
# ==============================================================================
def derive_primes_and_lambda(max_prime=13):
    is_prime = [True] * (max_prime + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(math.isqrt(max_prime)) + 1):
        if is_prime[p]:
            for i in range(p * p, max_prime + 1, p):
                is_prime[i] = False
    primes = [p for p, valid in enumerate(is_prime) if valid]

    product = 1.0
    for p in primes:
        product *= (1.0 - (1.0 / math.sqrt(p)))

    return primes, 1.0 - product

PRIME_ANCHORS, LAMBDA_DERIVED = derive_primes_and_lambda(max_prime=13)
print(f"Dynamically Derived Primes : {PRIME_ANCHORS}")
print(f"Dynamically Derived Lambda : {LAMBDA_DERIVED:.10f}")

# ==============================================================================
# 2. MoE Topological Governor
# ==============================================================================
class MoETopologicalGovernor:
    def __init__(self, prime_indices, lam):
        self.prime_indices = prime_indices
        self.lam = lam

    def pre_step(self, model: torch.nn.Module):
        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.grad is None:
                    continue
                param.grad.mul_(self.lam)
                if any(k in name.lower() for k in ["embed_tokens", "lm_head"]):
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid, :] = 0.0

    def post_step(self, model: torch.nn.Module, initial_anchors: dict):
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in initial_anchors:
                    param.data[self.prime_indices, :] = initial_anchors[name]

# ==============================================================================
# 3. Streaming Execution on Native MoE (Qwen1.5-MoE-A2.7B)
# ==============================================================================
def run_moe_stream(mode="topo", total_steps=1000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_id = "Qwen/Qwen1.5-MoE-A2.7B"

    print(f"\nInitializing {model_id} ({mode.upper()}) in Native bfloat16 on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=torch.bfloat16
    ).to(device)

    model.train()
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    target_embed_name = None
    target_embed_module = None
    for name, param in model.named_parameters():
        if "embed_tokens.weight" in name.lower():
            target_embed_name = name
            target_embed_module = param
            break

    if target_embed_name is None:
        raise ValueError("Target embedding tensor 'embed_tokens.weight' not located.")

    # Freeze base model parameters to prevent AdamW state blowup
    for param in model.parameters():
        param.requires_grad = False

    # Enable gradients strictly on the target embedding manifold
    target_embed_module.requires_grad = True

    # Snapshot initial prime coordinates
    initial_anchors = {
        target_embed_name: target_embed_module.data[PRIME_ANCHORS, :].clone().detach()
    }

    governor = MoETopologicalGovernor(PRIME_ANCHORS, LAMBDA_DERIVED) if mode == "topo" else None

    # AdamW allocates state strictly for embed_tokens (~200 MB)
    optimizer = torch.optim.AdamW([target_embed_module], lr=1e-5, weight_decay=1e-4)

    dataset = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True
    )

    def text_stream():
        for sample in dataset:
            txt = sample.get("text", "").strip()
            if len(txt) > 80:
                yield txt

    stream_iter = iter(text_stream())

    print(f"\n--- Commencing MoE Streaming [{mode.upper()}] ({total_steps:,} Steps | Seed {SEED}) ---")

    for step in range(1, total_steps + 1):
        batch = [next(stream_iter) for _ in range(2)]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        inputs["labels"] = inputs["input_ids"].clone()

        optimizer.zero_grad()
        loss = model(**inputs).loss
        loss.backward()

        if mode == "topo":
            governor.pre_step(model)
            optimizer.step()
            governor.post_step(model, initial_anchors)
        else:
            optimizer.step()

        if step == 1 or step % 100 == 0 or step == total_steps:
            with torch.no_grad():
                current_param = dict(model.named_parameters())[target_embed_name]
                drift = torch.max(
                    torch.abs(current_param[PRIME_ANCHORS, :] - initial_anchors[target_embed_name])
                ).item()
            print(f"Step {step:4d} / {total_steps} | MoE Loss: {loss.item():.4f} | Coordinate Drift: {drift:.10f}")

    return model, tokenizer

if __name__ == "__main__":
    moe_model, moe_tokenizer = run_moe_stream(mode="topo", total_steps=1000)

Dynamically Derived Primes : [2, 3, 5, 7, 11, 13]
Dynamically Derived Lambda : 0.9785142874

Initializing Qwen/Qwen1.5-MoE-A2.7B (TOPO) in Native bfloat16 on cuda...


Loading weights:   0%|          | 0/387 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]


--- Commencing MoE Streaming [TOPO] (1,000 Steps | Seed 123) ---


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step    1 / 1000 | MoE Loss: 2.6418 | Coordinate Drift: 0.0000000000
Step  100 / 1000 | MoE Loss: 2.5442 | Coordinate Drift: 0.0000000000
Step  200 / 1000 | MoE Loss: 2.6171 | Coordinate Drift: 0.0000000000
Step  300 / 1000 | MoE Loss: 2.2646 | Coordinate Drift: 0.0000000000
Step  400 / 1000 | MoE Loss: 2.3091 | Coordinate Drift: 0.0000000000
Step  500 / 1000 | MoE Loss: 3.2417 | Coordinate Drift: 0.0000000000
Step  600 / 1000 | MoE Loss: 2.6277 | Coordinate Drift: 0.0000000000
Step  700 / 1000 | MoE Loss: 2.0739 | Coordinate Drift: 0.0000000000
Step  800 / 1000 | MoE Loss: 2.4348 | Coordinate Drift: 0.0000000000
Step  900 / 1000 | MoE Loss: 1.9300 | Coordinate Drift: 0.0000000000
Step 1000 / 1000 | MoE Loss: 2.1642 | Coordinate Drift: 0.0000000000


## vision

In [1]:
import math
import random
import os
import gc
import numpy as np
import torch
import torchvision
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    set_seed
)
from PIL import Image

# Eliminate CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ============================================================================
# 0. Deterministic Setup (Seed 123)
# ============================================================================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

gc.collect()
torch.cuda.empty_cache()

# ============================================================================
# 1. Invariant Coordinate Ring Derivation
# ============================================================================
def derive_primes_and_lambda(max_prime=13):
    is_prime = [True] * (max_prime + 1)
    is_prime[0] = is_prime[1] = False
    for p in range(2, int(math.isqrt(max_prime)) + 1):
        if is_prime[p]:
            for i in range(p * p, max_prime + 1, p):
                is_prime[i] = False
    primes = [p for p, valid in enumerate(is_prime) if valid]

    product = 1.0
    for p in primes:
        product *= (1.0 - (1.0 / math.sqrt(p)))

    return primes, 1.0 - product

PRIME_ANCHORS, LAMBDA_DERIVED = derive_primes_and_lambda(max_prime=13)
print(f"Dynamically Derived Primes : {PRIME_ANCHORS}")
print(f"Dynamically Derived Lambda : {LAMBDA_DERIVED:.10f}")

# ============================================================================
# 2. Gemma-4 Multimodal Topological Governor
# ============================================================================
class Gemma4E4BTopologicalGovernor:
    def __init__(self, prime_indices, lam):
        self.prime_indices = prime_indices
        self.lam = lam

    def pre_step(self, model: torch.nn.Module):
        """Attenuate plastic gradients across visual and textual projections,
        and clamp prime coordinate gradients."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.grad is None:
                    continue
                param.grad.mul_(self.lam)
                # Shield shared embedding and projection coordinates
                if any(k in name.lower() for k in ["embed_tokens", "lm_head"]):
                    p_valid = [p for p in self.prime_indices if p < param.grad.shape[0]]
                    if p_valid:
                        param.grad[p_valid, :] = 0.0

    def post_step(self, model: torch.nn.Module, initial_anchors: dict):
        """Re-project prime coordinates against AdamW weight decay drift."""
        with torch.no_grad():
            for name, param in model.named_parameters():
                if name in initial_anchors:
                    param.data[self.prime_indices, :] = initial_anchors[name]

# ============================================================================
# 3. Model & Processor Initialization
# ============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "google/gemma-4-E4B"

print(f"\nInitializing Multimodal {model_id} (TOPO) in Native bfloat16 on {device}...")
processor = AutoProcessor.from_pretrained(model_id)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=torch.bfloat16
).to(device)

model.train()
model.config.use_cache = False

target_embed_name = None
target_embed_module = None
for name, param in model.named_parameters():
    if "embed_tokens.weight" in name.lower():
        target_embed_name = name
        target_embed_module = param
        break

if target_embed_name is None:
    raise ValueError("Target embedding tensor 'embed_tokens.weight' not located in Gemma-4-E4B architecture.")

for param in model.parameters():
    param.requires_grad = False

target_embed_module.requires_grad = True

initial_anchors = {
    target_embed_name: target_embed_module.data[PRIME_ANCHORS, :].clone().detach()
}

governor = Gemma4E4BTopologicalGovernor(PRIME_ANCHORS, LAMBDA_DERIVED)
optimizer = torch.optim.AdamW([target_embed_module], lr=1e-5, weight_decay=1e-4)

img_token = getattr(processor, "image_token", "<|image|>")

# ============================================================================
# 4. DATASET - STL-10 Setup
# ============================================================================
STL_CLASSES = {
    0: 'airplane', 1: 'bird', 2: 'car', 3: 'cat', 4: 'deer',
    5: 'dog', 6: 'horse', 7: 'monkey', 8: 'ship', 9: 'truck'
}

print(f"\n📚 LOADING STL-10 (PIL Raw format for multimodal ingestion)...")
trainset = torchvision.datasets.STL10(root='./data', split='train', download=True)
testset = torchvision.datasets.STL10(root='./data', split='test', download=True)

print(f"   Training set: {len(trainset):,} samples")
print(f"   Test set: {len(testset):,} samples")

# ============================================================================
# 5. 13 Continual Multimodal Tasks
# ============================================================================
def get_class_label(cls, task):
    return 0 if cls in task['class0'] else 1

TASKS_13 = {
    'A': {
        'name': 'Animal vs Vehicle',
        'class0': [1, 3, 4, 5, 6, 7],
        'class1': [0, 2, 8, 9],
        'label0_text': ['animal', 'living creature', 'wild animal'],
        'label1_text': ['vehicle', 'machine', 'transportation']
    },
    'B': {
        'name': 'Natural vs Man-Made',
        'class0': [1, 3, 4, 5, 6, 7],
        'class1': [0, 2, 8, 9],
        'label0_text': ['natural', 'organic', 'from nature'],
        'label1_text': ['man-made', 'artificial', 'human-built']
    },
    'C': {
        'name': 'Living vs Non-Living',
        'class0': [1, 3, 4, 5, 6, 7],
        'class1': [0, 2, 8, 9],
        'label0_text': ['living', 'alive', 'breathing'],
        'label1_text': ['non-living', 'inanimate', 'not alive']
    },
    'D': {
        'name': 'Large vs Small',
        'class0': [0, 2, 6, 8, 9],
        'class1': [1, 3, 4, 5, 7],
        'label0_text': ['large', 'big', 'large-sized'],
        'label1_text': ['small', 'tiny', 'small-sized']
    },
    'E': {
        'name': 'Ground vs Air/Water',
        'class0': [2, 3, 5, 6, 7],
        'class1': [0, 1, 4, 8, 9],
        'label0_text': ['ground', 'land-based', 'terrestrial'],
        'label1_text': ['air or water', 'non-terrestrial', 'flying/swimming']
    },
    'F': {
        'name': 'Domestic vs Wild',
        'class0': [2, 3, 5],
        'class1': [1, 4, 6, 7],
        'label0_text': ['domestic', 'tame', 'pet'],
        'label1_text': ['wild', 'untamed', 'savage']
    },
    'G': {
        'name': 'Mammal vs Non-Mammal',
        'class0': [3, 5, 6, 7],
        'class1': [0, 1, 2, 4, 8, 9],
        'label0_text': ['mammal', 'warm-blooded', 'fur-bearing'],
        'label1_text': ['non-mammal', 'cold-blooded', 'feathered/metal']
    },
    'H': {
        'name': 'Flying vs Non-Flying',
        'class0': [0, 1],
        'class1': [2, 3, 4, 5, 6, 7, 8, 9],
        'label0_text': ['flying', 'can fly', 'airborne'],
        'label1_text': ['non-flying', 'ground-based', 'earthbound']
    },
    'I': {
        'name': 'Fast vs Slow',
        'class0': [0, 2, 6, 8, 9],
        'class1': [1, 3, 4, 5, 7],
        'label0_text': ['fast-moving', 'quick', 'rapid'],
        'label1_text': ['slow-moving', 'slow', 'lethargic']
    },
    'J': {
        'name': 'Urban vs Rural',
        'class0': [0, 2, 8, 9],
        'class1': [1, 3, 4, 5, 6, 7],
        'label0_text': ['urban', 'city', 'man-made environment'],
        'label1_text': ['rural', 'countryside', 'natural environment']
    },
    'K': {
        'name': 'Predator vs Prey',
        'class0': [3, 5, 7],
        'class1': [1, 4, 6],
        'label0_text': ['predator', 'hunter', 'carnivore'],
        'label1_text': ['prey', 'herbivore', 'hunted']
    },
    'L': {
        'name': 'Nocturnal vs Diurnal',
        'class0': [3, 5, 7],
        'class1': [1, 4, 6],
        'label0_text': ['nocturnal', 'night-active', 'night'],
        'label1_text': ['diurnal', 'day-active', 'day']
    },
    'M': {
        'name': 'Domesticated vs Wild Animals',
        'class0': [3, 5],
        'class1': [1, 4, 6, 7],
        'label0_text': ['domesticated', 'pet', 'tame animal'],
        'label1_text': ['wild animal', 'untamed', 'free']
    },
}

TASK_ORDER = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M']

def create_vision_text(label, task_type):
    class_name = STL_CLASSES[label]
    task = TASKS_13[task_type]
    if label in task['class0']:
        choice_text = random.choice(task['label0_text'])
    else:
        choice_text = random.choice(task['label1_text'])
    return f"{img_token}A photographic image showing a {class_name}, which is categorized as {choice_text}."

def get_task_samples(dataset, class_list, num_samples, task_type):
    samples = []
    samples_per_class = num_samples // len(class_list)
    for cls in class_list:
        indices = [i for i, (_, label) in enumerate(dataset) if label == cls]
        available = min(len(indices), samples_per_class * 3)
        selected = random.sample(indices, available)
        for idx in selected:
            raw_img, cls_label = dataset[idx]
            # Ensure 48-divisible patch stride (96x96 -> 240x240)
            img_resized = raw_img.convert("RGB").resize((240, 240), resample=Image.Resampling.BICUBIC)
            text_prompt = create_vision_text(cls_label, task_type)
            samples.append((img_resized, text_prompt))
    random.shuffle(samples)
    return samples

# ============================================================================
# 6. Continual Learning Execution Across 13 Tasks
# ============================================================================
STEPS_PER_TASK = 100
total_stream_steps = 0

print(f"\n--- Commencing Gemma-4-E4B STL-10 Continual Task Execution [TOPO] ---")

for task_id in TASK_ORDER:
    task = TASKS_13[task_id]
    class_list = task['class0'] + task['class1']
    task_samples = get_task_samples(trainset, class_list, num_samples=600, task_type=task_id)

    print(f"\n▶ Entering Task {task_id}: {task['name']} ({len(task_samples)} samples prepared)")
    sample_idx = 0

    for step in range(1, STEPS_PER_TASK + 1):
        total_stream_steps += 1
        img, prompt = task_samples[sample_idx % len(task_samples)]
        sample_idx += 1

        inputs = processor(
            images=img,
            text=prompt,
            return_tensors="pt"
        ).to(device)

        inputs["labels"] = inputs["input_ids"].clone()

        optimizer.zero_grad()
        loss = model(**inputs).loss
        loss.backward()

        governor.pre_step(model)
        optimizer.step()
        governor.post_step(model, initial_anchors)

        if step == 1 or step % 50 == 0 or step == STEPS_PER_TASK:
            with torch.no_grad():
                current_param = dict(model.named_parameters())[target_embed_name]
                drift = torch.max(
                    torch.abs(current_param[PRIME_ANCHORS, :] - initial_anchors[target_embed_name])
                ).item()
            print(f"   [Task {task_id}] Step {step:3d}/{STEPS_PER_TASK} (Total {total_stream_steps:4d}) | Loss: {loss.item():.4f} | Drift: {drift:.10f}")

print("\n--- Continual Multimodal Training Across All 13 Tasks Concluded ---")

Dynamically Derived Primes : [2, 3, 5, 7, 11, 13]
Dynamically Derived Lambda : 0.9785142874

Initializing Multimodal google/gemma-4-E4B (TOPO) in Native bfloat16 on cuda...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]


📚 LOADING STL-10 (PIL Raw format for multimodal ingestion)...


100%|██████████| 2.64G/2.64G [11:24<00:00, 3.85MB/s]


   Training set: 5,000 samples
   Test set: 8,000 samples

--- Commencing Gemma-4-E4B STL-10 Continual Task Execution [TOPO] ---

▶ Entering Task A: Animal vs Vehicle (1800 samples prepared)
   [Task A] Step   1/100 (Total    1) | Loss: 31.4158 | Drift: 0.0000000000
   [Task A] Step  50/100 (Total   50) | Loss: 31.5279 | Drift: 0.0000000000
   [Task A] Step 100/100 (Total  100) | Loss: 32.3068 | Drift: 0.0000000000

▶ Entering Task B: Natural vs Man-Made (1800 samples prepared)
   [Task B] Step   1/100 (Total  101) | Loss: 28.0028 | Drift: 0.0000000000
   [Task B] Step  50/100 (Total  150) | Loss: 32.1468 | Drift: 0.0000000000
   [Task B] Step 100/100 (Total  200) | Loss: 28.8144 | Drift: 0.0000000000

▶ Entering Task C: Living vs Non-Living (1800 samples prepared)
   [Task C] Step   1/100 (Total  201) | Loss: 30.6959 | Drift: 0.0000000000
   [Task C] Step  50/100 (Total  250) | Loss: 29.4367 | Drift: 0.0000000000
   [Task C] Step 100/100 (Total  300) | Loss: 29.2655 | Drift: 0.0000000